# Master evaluation — every run scored identically

**The rule that makes this reviewable: no metric math lives in this notebook.** Every number
is a call into `pim.*` — probes from `pim.probes`, editors from `pim.editors`, scores from
`pim.metrics`, benches from `pim.environments.*` — so reviewing those packages *is* reviewing
these numbers.

**Since 2026-09-19 no wiring lives here either.** The scorers, the `scores.json` block schema,
the decodability floors and the add-only-what-is-missing bookkeeping are `pim/scoring/`, moved
verbatim from this notebook's cells. What stays here is what a reader needs to see:

| cell | what it does | where the code is |
|---|---|---|
| [1] | scan `runs/**/config.json` → the run table (honours `PIM_ONLY_RUNS`, `PIM_SKIP_TOPICS`; skips `archive/`, `_`-prefixed topics, runs still training) | `pim/scoring/runs.py` |
| [2] | **the canonical evaluation settings — every knob, in one visible place** (`PIM_DW_BASES`) + the version rule | this notebook |
| [3] | the two decodability floors per (instance, architecture) → `runs/_baselines/` | `pim/scoring/baselines.py` |
| [4] | score every run whose `scores.json` is missing, stale, or lacks a block / editor the settings ask of it | `pim/scoring/driver.py` → `discworld.py`, `othello.py`, `blocks.py` |
| [5] | per-run summaries, human-readable | `pim/scoring/summary.py` |

**What a `scores.json` holds.** Decodability (Probe Skill — the cross-environment axis — plus
native R² / error-rate, per residual point, with the MLP ≥ linear tripwire report), held-out
gates (Othello: legal mass, top-1, CE excess over the exact Bayes floor), and editability
(PI / ND / GS / IM / IM-NN: the full sweep plus each editor's best arm with its guards:
fidelity, collateral, li-error-vs-pre). The shape of one block is `pim.scoring.blocks.probe_block`.
Oracle editors and the nullspace probe/editor are a dedicated opt-in analysis, not part of
this default loop.

Companions: `build_paper_tables_and_figs.ipynb` / `build_full_tables.ipynb` read every
`scores.json` this writes and render the tables.

In [ ]:
# [1] Scan runs/ (recursively) -> the run table (pim.scoring.runs.scan_runs). Excluded by
#     convention: runs/archive/, any topic dir starting with "_", and a run STILL TRAINING
#     (its last logged step has not reached config train.steps). Two environment hooks:
#     PIM_ONLY_RUNS (a comma list of run names) and PIM_SKIP_TOPICS (a comma list of topics).
import json, os, sys, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)                       # repo-relative runs/ paths below; datasets/ paths come from
                                     # pim.environments.layout (REPO-anchored, layout v2 2026-09-10)
sys.path.insert(0, str(REPO))

from pim.scoring import scan_runs, score_all_baselines, score_all, print_summaries

RUNS = scan_runs()
print(f"{'topic':<28} {'run':<16} {'arch':<22} {'env':<10} {'instance':<12} scored")
for r in RUNS:
    print(f"{r['topic']:<28} {r['run']:<16} {r['arch']:<22} {r['env']:<10} "
          f"{r['instance']:<12} {r['scored']}")

In [ ]:
# [2] The canonical evaluation settings — every knob, in one visible place.
# Bump EVAL_VERSION to force a rescore of every run (a changed setting or a fixed bug);
# a run is skipped iff its scores.json carries this exact version AND holds every
# probe-target block the settings ask of it (a missing block is ADDED, nothing else is
# recomputed — cell [6]).
EVAL_VERSION = "2026-09-01.4"   # OTHELLO (and the default): discworld now fits ONE probe set
                                # per basis (the FULL state) and sweeps editability over BOTH
                                # dim sets, reporting the better. (.3 fitted pos and full
                                # probes separately and scored them as two blocks.)
# DISCWORLD has its own version since 2026-09-12: the write target became the PRE-dynamics
# state (pos[EF] − v·dt for the edited object, the current state for the rest — the state
# whose next frame is the stored edited frame; bench.bench_arrays). Probes are untouched;
# every discworld editor sweep is rescored; Othello never had the misalignment and is not.
# 2026-09-12 (evening, Sevan): ONE edit protocol in both environments — 1000 cases, a fixed edit position
# (discworld frame 20 / Othello a 20-move prefix), every instance its own bench (Othello: single-tile flips
# cut from the instance's own edits games; discworld: a selection of >= 2 differing rays), the FULL state
# written (dims "all" only), SHARED alpha grids and GS start layers, and the Othello headline moves to the
# symmetric-difference support (both indices stay in scores.json). Both environments are rescored.
EVAL_VERSION_BY_ENV = {"discworld": "2026-09-12.2", "othello": "2026-09-12.1"}
# the shared editor grids: one set for CATEGORICAL targets (Othello mine/theirs, discworld partitions),
# one for REGRESSION targets (discworld frustum full, Othello mine_signed); alpha is relative in all three
# editors (PI: multiple of the exact step; ND: fraction of the activation norm; GS: of the act scale)
ALPHA_CAT = {"pi": (0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0, 35.0, 60.0, 100.0),
             "nd": (0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0),
             "gs": (0.05, 0.1, 0.2, 0.35, 0.7, 1.5)}
ALPHA_REG = {"pi": (0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0, 35.0, 60.0, 100.0, 175.0),
             "nd": (0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0),
             "gs": (0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.35, 0.5, 0.7)}
GS_LAYERS = (0, 2, 4, 6, 8)      # GS start layers, both environments (the expensive editor)
def eval_version(r) -> str:
    return EVAL_VERSION_BY_ENV.get(r["env"], EVAL_VERSION)

SETTINGS = {
    # -- probes ----------------------------------------------------------------
    "dw_probe_seqs": 30_000,      # from the instance's probe split (120k available);
                                  # keeps MLP-128 at ~14 rows/param, out of memorisation
    "oth_probe_games": 20_000,    # the probe index range [91M, 91.02M)
    # -- discworld editability (the 2026-08-22 spec: both axes swept) ----------
    "dw_bench_n": 1000,
    # ONE probe target. The retired pos-only probe is not lost: for the LINEAR probe the
    # position rows of a full-state lstsq fit are BIT-IDENTICAL to a position-only fit
    # (multi-output least squares decomposes per output dim — verified on cached probes,
    # max|W_full[:4] − W_pos| = 0.0 in both bases), so dims="pos" reproduces it exactly.
    # The MLP does not decompose, so for GS the two dim sets are genuinely different
    # edits — which is why both are swept rather than one assumed. 2026-09-01.
    "dw_target": "full",
    "dw_edit_dims": ("all",),         # the FULL state is written (2026-09-12); "pos" retired
    # A basis is a different PROBE TARGET, hence its own scored block and table row.
    # 'frustum' = u = x/(scale*y) laterally, 1/y in depth (frustum.CANONICAL_DEPTH),
    # settled by the 2026-09-01 depth pilot: the inverse-depth family beats cartesian
    # on BOTH position and velocity, while y/rho beat it on position but LOSE on
    # velocity — coordinates the model cannot observe do not help.
    # 2026-09-11 (Sevan): cartesian DROPPED from new scoring — frustum has been the settled
    # basis since 2026-09-01 and the cartesian row had become a fixed cost (a second block,
    # a second floor set, a row per run in every table) with no reader. Runs scored before
    # this date keep their cartesian block in scores.json as a record; build_full_table hides
    # it (HIDDEN_BASES). No EVAL_VERSION bump: the scorer only ADDS blocks a run lacks.
    # 2026-09-19 (Sevan): Cartesian is actually our default basis for Discworld now, but keep both.
    # BOTH blocks are scored by default (was frustum only, cartesian on request). ⛔ frustum stays FIRST in
    # the tuple: a categorical target's probes are keyed under the instance's FIRST basis
    # (pim.scoring.blocks.discworld_blocks), so reordering would orphan every cached categorical probe and
    # skip those blocks on every newly scored run. Which basis a TABLE shows is its own knob (T.set_basis —
    # cartesian in the paper and appendix notebooks). PIM_DW_BASES still overrides (the queue jobs set it).
    # No EVAL_VERSION bump: every scored run and every floor file already carries both blocks (dry run 2026-09-19).
    "dw_bases": tuple(os.environ.get("PIM_DW_BASES", "frustum,cartesian").split(",")),
    # per-INSTANCE override (2026-09-13): dw-8ray-obs5 has five observers, so the frustum basis
    # (observer 0's lateral fraction + depth) is no longer privileged — its regression block and
    # its categorical probes are in world CARTESIAN coordinates. Every other instance: dw_bases.
    "dw_bases_by_instance": {"dw-8ray-obs5": ("cartesian",)},
    "dw_alpha_nd": ALPHA_REG["nd"], "dw_alpha_pi": ALPHA_REG["pi"], "dw_alpha_gs": ALPHA_REG["gs"],
    "gs_layers": GS_LAYERS,
    "dw_gs_steps": 100,
    "dw_gs_beta": 0.2,
    # -- extra discworld probe TARGETS, per run (2026-09-09) -------------------
    # A probe target is a scored BLOCK and a table ROW, exactly like a basis
    # (pim.environments.discworld.grid_target; findings/grid-target-control.md):
    #   grid-16x8      the state as 16 x 8 = 128 frustum-uniform cells x {empty, obj 0, obj 1}
    #                  (the noiseless run; probes fitted 2026-09-08, re-keyed 2026-09-09)
    #   appearance     the observation-exact partition of dw-8ray: a cell = the run of rays a
    #                  disc lights (30 cells); -d2/-d3 split each run into depth bands, -lat
    #                  merges runs by centre (15); grid-8x4 / -32x16 are the product grids
    #                  at other resolutions — the target-resolution sweep (2026-09-09→10)
    #   pos@<partition> a SNAPPED REGRESSION target (2026-09-10): the same gridification kept as
    #                  the 4-D position regression — every position replaced by the centre of
    #                  its cell (frustum basis), the regression probes / PI / GS unchanged, ND
    #                  not reported (continuous target), the bench filtered to cell-changing
    #                  teleports as for the partition itself. Fitted INLINE (the canonical 30k
    #                  regression recipe — minutes), floors inline like the canonical target.
    #   <partition>-fac the FACTORISED categorical target (2026-09-10): per object one softmax
    #                  per factor of the partition (appearance: run centre × run length —
    #                  2 × (15 + 5) classes on 4 tiles instead of 30 × 3), edits as per-tile
    #                  class swaps; the partition's cell-changing bench; GRID_PROBE_RECIPE,
    #                  fitted deliberately (scripts/drivers/probe_target_fit.sh).
    # ⛔ The scorer NEVER fits these (require_cached): a target whose probes are not yet in
    # the run's probes/ is SKIPPED and picked up on the next run of this notebook. Fit them
    # deliberately with scripts/fit_probes.py (recipe: arms.GRID_PROBE_RECIPE).
    "dw_extra_targets": {
        "noise_ablation/L-dw-noiseless-20m": ("grid-16x8", "grid-8x4", "appearance-lat",
                                             "grid-32x16",                  # chain 4, 2026-09-10 (grid-64x32 stopped, never fitted)
                                             "appearance-fac"),             # factorised, 2026-09-10 pm
        "blink_ablation/L-dw-blink-20m": ("appearance-fac",),                 # factorised, 2026-09-10 pm
        "smooth_ablation/L-dw-smooth-20m": ("appearance-fac",),               # anti-aliasing instance, 2026-09-12 night
        "observer_ablation/L-dw-8ray-obs5-20m": ("appearance-fac",),          # five observers: per-view factors, 2026-09-13
        "ray_ablation/L-dw-5ray-20m": ("appearance-fac", "appearance"),      # 5-ray instance, 2026-09-11
        "ray_ablation/L-dw-128ray-20m": ("appearance-fac",),                   # 128-ray at radius 1.0 (the ray axis closed), 2026-09-15 night
        "ray_ablation/R-dw-8ray-20m": ("appearance-fac",),                    # recurrent 8-ray, un-quarantined 2026-09-11
        "training_curve/L-dw-8ray-20m_s032000": ("appearance-fac",),          # loss-matched to the GRU's best (2026-09-11)
        # SEED REPLICATES (<parent>__seed<k>, config.json `replicate`) are NOT listed here: they
        # inherit their parent's extra targets (discworld_blocks → extra_targets_of, 2026-09-14).
        "ray_ablation/L-dw-8ray-20m": ("appearance", "appearance-d2", "appearance-d3",
                                       "grid-16x8", "appearance-lat", "grid-8x4", "grid-32x16",
                                       "grid-6x5", "grid-10x3", "grid-4x2",
                                       "pos@appearance",                     # snapped, 2026-09-10
                                       "appearance-fac"),                    # factorised, 2026-09-10
        "interface_ablation/L-dw-8ray-tok-20m": ("appearance", "appearance-d2", "appearance-d3",
                                                 "grid-16x8", "appearance-lat", "grid-8x4",
                                                 "grid-32x16", "grid-6x5", "grid-10x3", "grid-4x2",
                                                 "appearance-fac"),          # factorised, 2026-09-10 night
        "ray_ablation/L-dw-16ray-20m": ("appearance-fac",),
    },
    # The categorical targets' alpha grids. A categorical write needs bigger steps than the
    # regression one: on the canonical grids ND and GS pinned at their upper edge (ND best
    # at alpha 8, GS at 0.75), so these are the extended grids of the 2026-09-08 sweep.
    # Only dims "all" applies — a categorical target has no position/velocity read-outs.
    "dw_grid_alpha_pi": ALPHA_CAT["pi"], "dw_grid_alpha_nd": ALPHA_CAT["nd"], "dw_grid_alpha_gs": ALPHA_CAT["gs"],
    # -- othello editability (step-0 over the 1001-case bench) -----------------
    "oth_alpha_nd": ALPHA_CAT["nd"], "oth_alpha_pi": ALPHA_CAT["pi"], "oth_alpha_gs": ALPHA_CAT["gs"],
    "oth_gs_layers": GS_LAYERS,
    "oth_gs_steps": 100,
    "oth_gs_beta": 0.2,               # their reg_strg
    "oth_gates_games": 10_000,        # the whole test split
    # -- extra Othello probe TARGETS, every Othello run (2026-09-09) --------------
    # "mine_signed": the mine/theirs board as ONE SIGNED VALUE per tile (+1 mine, 0 blank,
    # −1 theirs), read by a 64-output REGRESSION probe — the same information in the same
    # frame as the canonical categorical target, so "categorical vs regression probe
    # target" is tested on Othello the way the grid target tests it on discworld. Fitted
    # inline by this notebook (minutes per run). Editors through the regression machinery:
    # PI sets the tile's read-out to ±1 (z-space, y-affine), ND adds the tile's probe row
    # with the sign of the flip (constant magnitude → sound), GS uses the MSE spec on the tile.
    "oth_extra_targets": ("mine_signed",),
    "oth_reg_alpha_pi": ALPHA_REG["pi"], "oth_reg_alpha_nd": ALPHA_REG["nd"], "oth_reg_alpha_gs": ALPHA_REG["gs"],
}
print(f"EVAL_VERSION {EVAL_VERSION}")

In [ ]:
# [2b] CATEGORICAL INVERSE MAPS (2026-09-20, Sevan) — which categorical discworld blocks get an IM arm.
#      The inverse map inverts THE STATE THE BLOCK'S OWN PROBES READ. A regression block: the continuous
#      full state of its basis (IM and IM-NN, every run). A categorical block: that target's own labels,
#      one-hot, PLUS the discs' continuous Cartesian velocity (deliberately asymmetric — the forward
#      probes read labels only, but a map from labels alone would write the mean residual over
#      velocities and erase the model's velocity estimate), fitted with the target's forward-probe
#      recipe (arms.GRID_PROBE_RECIPE: 200k sequences of probe_250k, 50 epochs, streamed), IM only.
#      Each such map is a ~30-minute fit per block, so it is computed only where the paper reports it;
#      every other categorical block carries NO IM arm and its table cell is blank. (Until this date a
#      categorical block's "IM" was its basis's CONTINUOUS map scored on the categorical bench — a state
#      its probes never read; those arms were removed from every scores.json.)
#      A run scored FRESH gets it. An already-scored run gets it ADDED only under PIM_ADD_CAT_IM=1
#      (pim.scoring.driver.missing_inverse) — one deliberate catch-up job, never a surprise inside
#      whichever replicate happens to score next.
SETTINGS["dw_cat_im"] = {"instances": ("dw-128ray", "dw-16ray", "dw-8ray", "dw-5ray"),
                         "targets": ("appearance-fac",)}

In [ ]:
# [3] BASELINES — the two decodability floors (observation, random-init), per environment
#     INSTANCE x ARCHITECTURE, in runs/_baselines/<instance>/baselines.json. Only what a file
#     lacks is fitted (a new architecture, a new basis, an extra target whose floor probes exist);
#     stamped with its own BASELINE_VERSION, independent of EVAL_VERSION.
#     Code + the full account of each floor: pim/scoring/baselines.py.
score_all_baselines(RUNS, SETTINGS)

In [ ]:
# [4] SCORE — every run whose scores.json is missing or stamped with a stale eval_version is
#     scored in full; a run that is current but lacks a probe-target block the SETTINGS ask of it,
#     or the IM editor, gets just that ADDED (nothing else recomputed; `blocks_added` /
#     `inverse_added` record when). One dated backup per eval version, atomic write.
#     Which scorer: by what the model emits and which world it lives in — frame model on discworld
#     (ray-zone Edit Index), frames-as-tokens on discworld (frame-set), Othello (legal-set).
#     Code: pim/scoring/driver.py → discworld.py / othello.py; one block's shape: blocks.probe_block.
score_all(RUNS, SETTINGS, eval_version)

In [ ]:
# [5] Per-run summaries: the headline block of each scores.json, human-readable
#     (pim/scoring/summary.py). Discworld: one block per probe target; Othello: the canonical
#     block, then every extra-target block.
print_summaries(RUNS)